# Synthetic Banking A/B Testing — Experiment Design

This notebook validates three pre-registered experiments before treatment effects are interpreted.

It covers the experiment catalog, assignment integrity, pre-treatment balance, power, minimum detectable effects, and decision readiness.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import yaml

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from experiment_statistics import (
    balance_table,
    required_sample_size_continuous,
    required_sample_size_proportion,
    srm_test,
)

with (ROOT / "config" / "experiment_config.yml").open(encoding="utf-8") as handle:
    CONFIG = yaml.safe_load(handle)

PROJECT = CONFIG["project"]
EXPERIMENTS = CONFIG["experiments"]

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

## Load experiment datasets

Run `python src/build_experiment_data.py` before executing this notebook.

In [2]:
datasets = {}

for experiment_id, experiment in EXPERIMENTS.items():
    path = ROOT / experiment["output_file"]
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path.relative_to(ROOT)}. "
            "Run python src/build_experiment_data.py first."
        )
    datasets[experiment_id] = pd.read_csv(path)

{experiment_id: frame.shape for experiment_id, frame in datasets.items()}

{'EXP-01': (50000, 19), 'EXP-02': (3428, 14), 'EXP-03': (3087, 17)}

## Pre-registered experiment catalog

In [3]:
catalog = pd.DataFrame(
    [
        {
            "experiment_id": experiment_id,
            "experiment": experiment["name"],
            "primary_metric": experiment["primary_metric_label"],
            "sample_cap": experiment["sample_cap"],
            "strata": ", ".join(experiment["strata"]),
            "pre_period_days": experiment["pre_period_days"],
            "experiment_days": experiment["experiment_days"],
            "random_seed": experiment["random_seed"],
        }
        for experiment_id, experiment in EXPERIMENTS.items()
    ]
)
catalog

,experiment_id,experiment,primary_metric,sample_cap,strata,pre_period_days,experiment_days,random_seed
0,EXP-01,Live FX spread discount,Revenue per assigned customer,50000,"segment_code, dominant_channel_code",90,30,20261729
1,EXP-02,DCD cross-sell outreach,DCD conversion rate,40000,segment_code,180,45,20262729
2,EXP-03,Digital channel migration,Digital adoption rate,50000,"segment_code, dominant_assisted_channel_code",90,90,20263729


## Assignment integrity

Sample ratio mismatch is checked before any outcome comparison. A significant result invalidates causal interpretation until the assignment or data pipeline is investigated.

In [4]:
srm_rows = []

for experiment_id, frame in datasets.items():
    result = srm_test(
        frame["variant"],
        expected_treatment_share=PROJECT["expected_treatment_share"],
    )
    result["experiment_id"] = experiment_id
    result["srm_passed"] = result["p_value"] >= PROJECT["alpha"]
    srm_rows.append(result)

srm_summary = pd.DataFrame(srm_rows)[
    [
        "experiment_id",
        "control_n",
        "treatment_n",
        "treatment_share",
        "chi_square",
        "p_value",
        "srm_passed",
    ]
]
srm_summary

,experiment_id,control_n,treatment_n,treatment_share,chi_square,p_value,srm_passed
0,EXP-01,24999,25001,0.5000,0.0001,0.9929,True
1,EXP-02,1714,1714,0.5000,0.0000,1.0000,True
2,EXP-03,1544,1543,0.4998,0.0003,0.9856,True


## Pre-treatment numeric balance

In [5]:
balance_columns = {
    "EXP-01": [
        "prior_fx_transactions",
        "prior_fx_turnover_usd",
        "prior_fx_revenue_usd",
        "prior_avg_spread",
    ],
    "EXP-02": [
        "prior_fx_transactions",
        "prior_fx_turnover_usd",
        "prior_fx_revenue_usd",
        "historical_dcd_conversion_rate",
    ],
    "EXP-03": [
        "prior_fx_transactions",
        "prior_assisted_transactions",
        "prior_digital_share",
        "prior_fx_revenue_usd",
        "historical_digital_adoption_rate",
    ],
}

balance_frames = []

for experiment_id, columns in balance_columns.items():
    table = balance_table(datasets[experiment_id], columns)
    table.insert(0, "experiment_id", experiment_id)
    table["absolute_smd"] = table["standardised_mean_difference"].abs()
    table["balance_passed"] = table["absolute_smd"] < 0.10
    balance_frames.append(table)

balance_summary = pd.concat(balance_frames, ignore_index=True)
balance_summary

,experiment_id,covariate,control_mean,treatment_mean,standardised_mean_difference,absolute_smd,balance_passed
0,EXP-01,prior_fx_transactions,12.6636,12.7362,0.0073,0.0073,True
1,EXP-01,prior_fx_turnover_usd,"14,852,791.1535","14,307,581.2632",-0.0130,0.0130,True
2,EXP-01,prior_fx_revenue_usd,"27,169.2900","26,233.3090",-0.0116,0.0116,True
3,EXP-01,prior_avg_spread,0.0043,0.0043,-0.0085,0.0085,True
4,EXP-02,prior_fx_transactions,19.7077,19.2340,-0.0231,0.0231,True
5,EXP-02,prior_fx_turnover_usd,"23,130,854.6713","21,650,655.4828",-0.0279,0.0279,True
6,EXP-02,prior_fx_revenue_usd,"38,127.7898","36,421.5081",-0.0186,0.0186,True
7,EXP-02,historical_dcd_conversion_rate,0.1357,0.1357,0.0000,0.0000,True
8,EXP-03,prior_fx_transactions,10.5278,10.5172,-0.0011,0.0011,True
9,EXP-03,prior_assisted_transactions,6.4197,6.4083,-0.0022,0.0022,True


## Power and required sample size

EXP-01 uses a pre-specified standardised MDE. EXP-02 and EXP-03 use historical pre-experiment baseline rates exported by SQL, not observed experiment outcomes.

In [6]:
power_rows = []

fx_config = EXPERIMENTS["EXP-01"]
fx_counts = datasets["EXP-01"]["variant"].value_counts()
power_rows.append(
    {
        "experiment_id": "EXP-01",
        "metric": fx_config["primary_metric_label"],
        "baseline": None,
        "mde": fx_config["continuous_mde_standardised"],
        "mde_type": "standardised",
        "required_per_arm": required_sample_size_continuous(
            fx_config["continuous_mde_standardised"],
            alpha=PROJECT["alpha"],
            power=PROJECT["target_power"],
        ),
        "available_control": int(fx_counts.get("control", 0)),
        "available_treatment": int(fx_counts.get("treatment", 0)),
    }
)

binary_design = {
    "EXP-02": "historical_dcd_conversion_rate",
    "EXP-03": "historical_digital_adoption_rate",
}

for experiment_id, baseline_column in binary_design.items():
    experiment = EXPERIMENTS[experiment_id]
    frame = datasets[experiment_id]
    baseline = frame[baseline_column].clip(0.001, 0.999).mean()
    counts = frame["variant"].value_counts()
    power_rows.append(
        {
            "experiment_id": experiment_id,
            "metric": experiment["primary_metric_label"],
            "baseline": baseline,
            "mde": experiment["proportion_mde_absolute"],
            "mde_type": "absolute proportion",
            "required_per_arm": required_sample_size_proportion(
                baseline,
                experiment["proportion_mde_absolute"],
                alpha=PROJECT["alpha"],
                power=PROJECT["target_power"],
            ),
            "available_control": int(counts.get("control", 0)),
            "available_treatment": int(counts.get("treatment", 0)),
        }
    )

power_summary = pd.DataFrame(power_rows)
power_summary["sample_requirement_met"] = (
    power_summary[["available_control", "available_treatment"]].min(axis=1)
    >= power_summary["required_per_arm"]
)
power_summary

,experiment_id,metric,baseline,mde,mde_type,required_per_arm,available_control,available_treatment,sample_requirement_met
0,EXP-01,Revenue per assigned customer,NaN,0.0400,standardised,9813,24999,25001,True
1,EXP-02,DCD conversion rate,0.1357,0.0150,absolute proportion,8555,1714,1714,False
2,EXP-03,Digital adoption rate,0.7644,0.0300,absolute proportion,2996,1544,1543,False


## Design readiness

In [7]:
balance_readiness = (
    balance_summary.groupby("experiment_id", as_index=False)["balance_passed"]
    .all()
    .rename(columns={"balance_passed": "all_balance_checks_passed"})
)

design_summary = (
    srm_summary[["experiment_id", "srm_passed"]]
    .merge(balance_readiness, on="experiment_id", how="left")
    .merge(power_summary, on="experiment_id", how="left")
)
design_summary["analysis_valid"] = design_summary[
    ["srm_passed", "all_balance_checks_passed"]
].all(axis=1)
design_summary["decision_ready"] = design_summary[
    ["analysis_valid", "sample_requirement_met"]
].all(axis=1)
design_summary

,experiment_id,srm_passed,all_balance_checks_passed,metric,baseline,mde,mde_type,required_per_arm,available_control,available_treatment,sample_requirement_met,analysis_valid,decision_ready
0,EXP-01,True,True,Revenue per assigned customer,NaN,0.0400,standardised,9813,24999,25001,True,True,True
1,EXP-02,True,True,DCD conversion rate,0.1357,0.0150,absolute proportion,8555,1714,1714,False,True,False
2,EXP-03,True,True,Digital adoption rate,0.7644,0.0300,absolute proportion,2996,1544,1543,False,True,False


In [8]:
results_path = ROOT / "results" / "design_summary.csv"
results_path.parent.mkdir(parents=True, exist_ok=True)
design_summary.to_csv(results_path, index=False, encoding="utf-8-sig")
results_path.relative_to(ROOT)

WindowsPath('results/design_summary.csv')

## Design conclusion

`analysis_valid` confirms assignment integrity and acceptable pre-treatment balance. `decision_ready` additionally confirms that the available sample meets the pre-specified power requirement. Underpowered experiments may be inspected, but their product decision remains `INCONCLUSIVE`.